# Advanced Solution: Winner's Methodology

## Подход победителя

1. **10-15 моделей** (XGB/LGB/Cat вариации)
2. **RepeatedStratifiedKFold** (3x5) для всех моделей
3. **Ridge Blending** (alpha=0.001, fit_intercept=False)
4. **Группировка моделей** (мета-признаки)
5. **Permutation Importance** для отбора моделей
6. **Иерархическая кластеризация**
7. **A/B тесты** (Feature Jittering, Label Smoothing)

**Цель:** R² ≥ 0.869

In [1]:
# Импорт
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
import warnings
import os
from time import time

# ML
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
from scipy.stats.mstats import winsorize
from scipy.cluster import hierarchy
from scipy.spatial.distance import squareform

# Gradient Boosting
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
np.random.seed(42)

print('='*70)
print('ADVANCED SOLUTION - WINNER METHODOLOGY')
print('='*70)
print('✅ Библиотеки загружены')

ADVANCED SOLUTION - WINNER METHODOLOGY
✅ Библиотеки загружены


In [2]:
# Загрузка данных
train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')

X_full = train.drop(columns=['id', 'FloodProbability'])
y_full = train['FloodProbability']
X_test = test.drop(columns=['id'])
test_ids = test['id']

original_features = X_full.columns.tolist()

print(f'Train: {train.shape}, Test: {test.shape}')
print(f'Target range: [{y_full.min():.4f}, {y_full.max():.4f}]')

Train: (1117957, 22), Test: (745305, 21)
Target range: [0.2850, 0.7250]


In [3]:
# Загрузка Baseline predictions (ИНКРЕМЕНТ!)
try:
    baseline_train_oof = np.load('baseline_output/baseline_predictions_train_oof.npy')
    baseline_test = np.load('baseline_output/baseline_predictions_test.npy')
    baseline_r2 = r2_score(y_full, baseline_train_oof)
    print(f'✅ Baseline loaded: R² = {baseline_r2:.6f}')
    BASELINE_AVAILABLE = True
except FileNotFoundError:
    print('⚠️ Baseline не найден. Запустите baseline_xgb.ipynb')
    BASELINE_AVAILABLE = False

✅ Baseline loaded: R² = 0.858277


In [4]:
# Винсоризация + Масштабирование
def preprocess(df, scaler=None, fit=False):
    df_win = df.copy()
    for col in df.columns:
        df_win[col] = winsorize(df[col], limits=(0.01, 0.01))
    
    if fit:
        scaler = RobustScaler()
        df_scaled = pd.DataFrame(scaler.fit_transform(df_win), columns=df.columns, index=df.index)
        return df_scaled, scaler
    else:
        df_scaled = pd.DataFrame(scaler.transform(df_win), columns=df.columns, index=df.index)
        return df_scaled

X_full_scaled, scaler = preprocess(X_full, fit=True)
X_test_scaled = preprocess(X_test, scaler=scaler)

print('✅ Preprocessing done')

✅ Preprocessing done


In [5]:
# Feature Engineering (расширенный набор от победителя)
def generate_features(df_orig, df_scaled, y=None, is_train=True):
    features = df_scaled.copy()
    
    print('Generating features...')
    
    # Базовые статистики
    features['sum'] = df_orig.sum(axis=1)
    features['mean'] = df_orig.mean(axis=1)
    features['std'] = df_orig.std(axis=1)
    features['max'] = df_orig.max(axis=1)
    features['min'] = df_orig.min(axis=1)
    features['median'] = df_orig.median(axis=1)
    features['range'] = features['max'] - features['min']
    features['q25'] = df_orig.quantile(0.25, axis=1)
    features['q75'] = df_orig.quantile(0.75, axis=1)
    features['iqr'] = features['q75'] - features['q25']
    features['cv'] = features['std'] / (features['mean'] + 1e-10)
    
    # Count features (от победителя!)
    for threshold in [6, 7, 8]:
        features[f'nb_sup{threshold}'] = (df_orig > threshold).sum(axis=1)
    for threshold in [2, 3, 4]:
        features[f'nb_inf{threshold}'] = (df_orig < threshold).sum(axis=1)
    
    # Sorted features (от победителя!)
    print('  Creating sorted features...')
    sorted_vals = np.sort(df_orig.values, axis=1)
    for i in range(sorted_vals.shape[1]):
        features[f'sorted_{i}'] = sorted_vals[:, i]
    
    # Magic feature (от победителя!)
    if is_train and y is not None:
        temp_df = features[['sum']].copy()
        temp_df['target'] = y
        features['magic_std'] = temp_df.groupby('sum')['target'].transform('std')
        features['magic_std'].fillna(features['magic_std'].mean(), inplace=True)
    else:
        features['magic_std'] = 0
    
    print(f'  Total features: {features.shape[1]}')
    return features

X_full_feat = generate_features(X_full, X_full_scaled, y_full, is_train=True)
X_test_feat = generate_features(X_test, X_test_scaled, is_train=False)

# Заполняем magic_std для test
X_test_feat['magic_std'] = X_full_feat['magic_std'].mean()

print(f'\n✅ Features: {X_full_feat.shape[1]} (исходные {len(original_features)} → +{X_full_feat.shape[1] - len(original_features)})')

Generating features...
  Creating sorted features...
  Total features: 58
Generating features...
  Creating sorted features...
  Total features: 58

✅ Features: 58 (исходные 20 → +38)


In [6]:
# RepeatedStratifiedKFold setup (как у победителя)
n_repeats, n_splits, seed = 3, 5, 42
rskf = RepeatedStratifiedKFold(n_splits=n_splits, n_repeats=n_repeats, random_state=seed)

# Стратификация по дискретизованному таргету
y_discrete = (y_full * 400).astype(np.int16)

print(f'CV: RepeatedStratifiedKFold (n_repeats={n_repeats}, n_splits={n_splits})')
print(f'Total folds: {n_repeats * n_splits}')
print(f'Target discretization: (target * 400).astype(int16)')

CV: RepeatedStratifiedKFold (n_repeats=3, n_splits=5)
Total folds: 15
Target discretization: (target * 400).astype(int16)


In [7]:
# A/B тест: Feature Jittering & Label Smoothing
from sklearn.model_selection import train_test_split

print('='*70)
print('A/B ТЕСТ: Feature Jittering & Label Smoothing')
print('='*70)

X_train_ab, X_val_ab, y_train_ab, y_val_ab = train_test_split(
    X_full_feat, y_full, test_size=0.2, random_state=42
)

# Базовая модель
xgb_test = XGBRegressor(n_estimators=50, max_depth=5, random_state=42, n_jobs=-1)
xgb_test.fit(X_train_ab, y_train_ab)
r2_clean = r2_score(y_val_ab, xgb_test.predict(X_val_ab))

# Jittering
np.random.seed(42)
X_train_jitter = X_train_ab + np.random.normal(0, 0.01, X_train_ab.shape)
y_train_smooth = y_train_ab * 0.95 + y_train_ab.mean() * 0.05

xgb_aug = XGBRegressor(n_estimators=50, max_depth=5, random_state=42, n_jobs=-1)
xgb_aug.fit(X_train_jitter, y_train_smooth)
r2_aug = r2_score(y_val_ab, xgb_aug.predict(X_val_ab))

print(f'\nClean:      R² = {r2_clean:.6f}')
print(f'Augmented:  R² = {r2_aug:.6f}')
print(f'Δ = {r2_aug - r2_clean:+.6f}')

USE_AUGMENTATION = r2_aug > r2_clean
print(f'\n{'✅ Используем аугментацию' if USE_AUGMENTATION else '❌ Аугментация не помогает'}')
print('='*70)

A/B ТЕСТ: Feature Jittering & Label Smoothing

Clean:      R² = 0.868756
Augmented:  R² = 0.865220
Δ = -0.003537

❌ Аугментация не помогает


In [8]:
# Функция обучения с RepeatedStratifiedKFold
def train_model_cv(model, X, y, model_name, use_aug=False):
    print(f'\nОбучение: {model_name}...')
    start = time()
    
    oof_preds = np.zeros(len(X))
    test_preds = np.zeros(len(X_test_feat))
    fold_scores = []
    
    for fold_idx, (train_idx, val_idx) in enumerate(rskf.split(X, y_discrete)):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
        
        # Аугментация если нужно
        if use_aug:
            np.random.seed(fold_idx)
            X_train = X_train + np.random.normal(0, 0.01, X_train.shape)
            y_train = y_train * 0.95 + y_train.mean() * 0.05
        
        # Обучение
        model.fit(X_train, y_train)
        
        # OOF
        oof_preds[val_idx] = model.predict(X_val)
        test_preds += model.predict(X_test_feat) / (n_repeats * n_splits)
        
        # Score
        fold_r2 = r2_score(y_val, oof_preds[val_idx])
        fold_scores.append(fold_r2)
    
    oof_r2 = r2_score(y, oof_preds)
    elapsed = time() - start
    
    print(f'  OOF R² = {oof_r2:.6f} (mean fold: {np.mean(fold_scores):.6f}, std: {np.std(fold_scores):.6f})')
    print(f'  Time: {elapsed:.1f}s')
    
    return oof_preds, test_preds, oof_r2

print('✅ Функция обучения готова')

✅ Функция обучения готова


In [ ]:
# Обучение ансамбля моделей (10-15 вариаций)
print('='*70)
print('ОБУЧЕНИЕ АНСАМБЛЯ МОДЕЛЕЙ (10-15 вариаций)')
print('='*70)

models_oofs = {}
models_preds = {}
models_scores = {}

# XGBoost вариации
for i, params in enumerate([
    {'n_estimators': 200, 'max_depth': 5, 'learning_rate': 0.05, 'subsample': 0.8, 'colsample_bytree': 0.8},
    {'n_estimators': 300, 'max_depth': 7, 'learning_rate': 0.03, 'subsample': 0.7, 'colsample_bytree': 0.9},
    {'n_estimators': 250, 'max_depth': 6, 'learning_rate': 0.05, 'subsample': 0.9, 'colsample_bytree': 0.7},
], 1):
    model = XGBRegressor(**params, random_state=42, n_jobs=-1, tree_method='hist', verbosity=0)
    oof, pred, score = train_model_cv(model, X_full_feat, y_full, f'xgb{i}', use_aug=USE_AUGMENTATION)
    models_oofs[f'xgb{i}'] = oof
    models_preds[f'xgb{i}'] = pred
    models_scores[f'xgb{i}'] = score

# LightGBM вариации
for i, params in enumerate([
    {'n_estimators': 200, 'num_leaves': 31, 'learning_rate': 0.05, 'subsample': 0.8, 'colsample_bytree': 0.8},
    {'n_estimators': 300, 'num_leaves': 63, 'learning_rate': 0.03, 'subsample': 0.7, 'colsample_bytree': 0.9},
    {'n_estimators': 250, 'num_leaves': 47, 'learning_rate': 0.05, 'subsample': 0.9, 'colsample_bytree': 0.7, 'boosting_type': 'dart'},
], 1):
    model = LGBMRegressor(**params, random_state=42, n_jobs=-1, verbose=-1)
    oof, pred, score = train_model_cv(model, X_full_feat, y_full, f'lgbm{i}', use_aug=USE_AUGMENTATION)
    models_oofs[f'lgbm{i}'] = oof
    models_preds[f'lgbm{i}'] = pred
    models_scores[f'lgbm{i}'] = score

# CatBoost вариации
for i, params in enumerate([
    {'iterations': 200, 'depth': 6, 'learning_rate': 0.05, 'subsample': 0.8},
    {'iterations': 300, 'depth': 7, 'learning_rate': 0.03, 'subsample': 0.7},
    {'iterations': 250, 'depth': 8, 'learning_rate': 0.05, 'subsample': 0.9},
    {'iterations': 200, 'depth': 6, 'learning_rate': 0.05, 'subsample': 0.8, 'bagging_temperature': 1.0},
], 1):
    model = CatBoostRegressor(**params, random_seed=42, verbose=0, task_type='CPU')
    oof, pred, score = train_model_cv(model, X_full_feat, y_full, f'cat{i}', use_aug=USE_AUGMENTATION)
    models_oofs[f'cat{i}'] = oof
    models_preds[f'cat{i}'] = pred
    models_scores[f'cat{i}'] = score

print(f'\n✅ Всего моделей обучено: {len(models_oofs)}')
print('='*70)

ОБУЧЕНИЕ АНСАМБЛЯ МОДЕЛЕЙ (10-15 вариаций)

Обучение: xgb1...
  OOF R² = 0.868832 (mean fold: 0.868829, std: 0.000335)
  Time: 45.2s

Обучение: xgb2...
  OOF R² = 0.869285 (mean fold: 0.869284, std: 0.000344)
  Time: 84.6s

Обучение: xgb3...


In [ ]:
# Добавляем baseline predictions (ИНКРЕМЕНТ!)
if BASELINE_AVAILABLE:
    models_oofs['baseline'] = baseline_train_oof
    models_preds['baseline'] = baseline_test
    models_scores['baseline'] = baseline_r2
    print(f'✅ Baseline добавлен в ансамбль: R² = {baseline_r2:.6f}')

In [ ]:
# Иерархическая кластеризация (от победителя!)
print('\n' + '='*70)
print('ИЕРАРХИЧЕСКАЯ КЛАСТЕРИЗАЦИЯ')
print('='*70)

oofs_df = pd.DataFrame(models_oofs)
correlations = oofs_df.corr()

print('\nКорреляция моделей:')
print(correlations)

# Dendrogram
fig, ax = plt.subplots(figsize=(15, 8))
converted_corr = 1 - np.abs(correlations)
Z = hierarchy.linkage(squareform(converted_corr), 'complete')
hierarchy.dendrogram(Z, labels=oofs_df.columns, ax=ax, orientation='right')
ax.set_title('Hierarchical Clustering of Models', fontsize=14, fontweight='bold')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('advanced_hierarchical_clustering.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n✅ Кластеризация завершена')
print('='*70)

In [ ]:
# Группировка моделей (мета-признаки, как у победителя!)
print('\n' + '='*70)
print('ГРУППИРОВКА МОДЕЛЕЙ')
print('='*70)

# Автоматическая группировка по высокой корреляции
threshold = 0.95
grouped = []
for i, col1 in enumerate(correlations.columns):
    for j, col2 in enumerate(correlations.columns):
        if i < j and correlations.iloc[i, j] > threshold:
            grouped.append((col1, col2, correlations.iloc[i, j]))

print(f'\nМодели с корреляцией > {threshold}:')
for g in grouped:
    print(f'  {g[0]} ↔ {g[1]}: {g[2]:.4f}')

# Создаем grouped модели
if len(grouped) > 0:
    print('\nСоздаем grouped признаки...')
    # Пример: группируем похожие модели
    for g in grouped[:3]:  # Топ-3 пары
        name = f'{g[0]}_{g[1]}_grouped'
        models_oofs[name] = (models_oofs[g[0]] + models_oofs[g[1]]) / 2
        models_preds[name] = (models_preds[g[0]] + models_preds[g[1]]) / 2
        print(f'  ✅ {name}')
else:
    print('\n⚠️ Нет моделей с высокой корреляцией для группировки')

print('='*70)

In [ ]:
# Ridge Blending (как у победителя!)
print('\n' + '='*70)
print('RIDGE BLENDING (alpha=0.001, fit_intercept=False)')
print('='*70)

# Собираем все OOF predictions
blend_X = np.column_stack([models_oofs[m] for m in models_oofs.keys()])
blend_test = np.column_stack([models_preds[m] for m in models_preds.keys()])

print(f'\nBlend matrix: {blend_X.shape}')
print(f'Models: {list(models_oofs.keys())}')

# Ridge (как у победителя: alpha=0.001, fit_intercept=False)
ridge = Ridge(alpha=0.001, fit_intercept=False, random_state=42)
ridge.fit(blend_X, y_full)

# Predictions
ridge_oof = ridge.predict(blend_X)
ridge_test = ridge.predict(blend_test)
ridge_r2 = r2_score(y_full, ridge_oof)

print(f'\n🏆 Ridge Blend R² = {ridge_r2:.6f}')

# Коэффициенты
print(f'\nВеса моделей:')
coefs_df = pd.DataFrame({
    'Model': models_oofs.keys(),
    'Coefficient': ridge.coef_
}).sort_values('Coefficient', ascending=False)

print(coefs_df.to_string(index=False))

# Визуализация
fig, ax = plt.subplots(figsize=(12, 8))
ax.barh(coefs_df['Model'], coefs_df['Coefficient'])
ax.axvline(0, color='red', linestyle='--', linewidth=2)
ax.set_xlabel('Coefficient', fontsize=12, fontweight='bold')
ax.set_title('Ridge Blending Weights', fontsize=14, fontweight='bold')
ax.grid(alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig('advanced_ridge_weights.png', dpi=150, bbox_inches='tight')
plt.show()

print('='*70)

In [ ]:
# Permutation Importance (для моделей в ансамбле)
print('\n' + '='*70)
print('PERMUTATION IMPORTANCE')
print('='*70)

n_permutations = 5
perm_scores = {}

print(f'\nВычисление importance (n_permutations={n_permutations})...')

baseline_score = r2_score(y_full, ridge_oof)

for model_name in tqdm(models_oofs.keys(), desc='Models'):
    scores = []
    for _ in range(n_permutations):
        # Перемешиваем predictions модели
        blend_X_perm = blend_X.copy()
        model_idx = list(models_oofs.keys()).index(model_name)
        blend_X_perm[:, model_idx] = np.random.permutation(blend_X_perm[:, model_idx])
        
        # Пересчитываем Ridge
        perm_pred = ridge.predict(blend_X_perm)
        perm_score = r2_score(y_full, perm_pred)
        
        # Importance = падение качества
        scores.append(baseline_score - perm_score)
    
    perm_scores[model_name] = np.mean(scores)

# Результаты
perm_df = pd.DataFrame({
    'Model': perm_scores.keys(),
    'Importance': perm_scores.values()
}).sort_values('Importance', ascending=False)

print(f'\nPermutation Importance (топ-10):')
print(perm_df.head(10).to_string(index=False))

# Визуализация
fig, ax = plt.subplots(figsize=(12, 8))
perm_df_plot = perm_df.head(15)
ax.barh(perm_df_plot['Model'], perm_df_plot['Importance'], color='green')
ax.set_xlabel('Importance (R² drop)', fontsize=12, fontweight='bold')
ax.set_title('Permutation Importance (топ-15)', fontsize=14, fontweight='bold')
ax.grid(alpha=0.3, axis='x')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('advanced_permutation_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('='*70)

In [ ]:
# Итоговая таблица результатов
print('\n' + '='*70)
print('ИТОГОВАЯ ТАБЛИЦА')
print('='*70)

results_df = pd.DataFrame({
    'Model': list(models_scores.keys()) + ['Ridge_Blend'],
    'R² (OOF)': list(models_scores.values()) + [ridge_r2]
}).sort_values('R² (OOF)', ascending=False)

print('\n' + results_df.to_string(index=False))

# Лучшая модель
best_model = results_df.iloc[0]['Model']
best_r2 = results_df.iloc[0]['R² (OOF)']

print(f'\n{'='*70}')
print(f'🏆 ЛУЧШАЯ МОДЕЛЬ: {best_model}')
print(f'🏆 R² = {best_r2:.6f}')

if BASELINE_AVAILABLE:
    improvement = best_r2 - baseline_r2
    improvement_pct = (improvement / baseline_r2) * 100
    print(f'\n📊 ИНКРЕМЕНТ:')
    print(f'   Baseline:  R² = {baseline_r2:.6f}')
    print(f'   Advanced:  R² = {best_r2:.6f}')
    print(f'   Δ R²:      {improvement:+.6f}')
    print(f'   Δ %:       {improvement_pct:+.2f}%')

print('='*70)

In [ ]:
# Submission (используем Ridge Blend)
submission = pd.DataFrame({
    'id': test_ids,
    'FloodProbability': ridge_test
})

submission.to_csv('submission_advanced_winner.csv', index=False)

print(f'\n✅ Submission: submission_advanced_winner.csv')
print(f'   R² (OOF): {ridge_r2:.6f}')
print(f'   Predictions range: [{ridge_test.min():.4f}, {ridge_test.max():.4f}]')

In [ ]:
print('\n' + '='*70)
print('🎯 FINAL SUMMARY')
print('='*70)

print('\n📋 РЕАЛИЗОВАННЫЕ ТЕХНИКИ ПОБЕДИТЕЛЯ:')
print('   ✅ RepeatedStratifiedKFold (3x5)')
print('   ✅ Стратификация по (target*400).astype(int16)')
print(f'   ✅ {len(models_oofs)} моделей (XGB/LGB/Cat вариации)')
print('   ✅ Ridge(alpha=0.001, fit_intercept=False)')
print('   ✅ Группировка моделей (мета-признаки)')
print('   ✅ Permutation Importance')
print('   ✅ Иерархическая кластеризация')
print('   ✅ A/B тесты (Jittering, Label Smoothing)')
print('   ✅ Инкремент (baseline в ансамбле)')

print('\n📋 ИЗ ТРЕБОВАНИЙ ЗАДАНИЯ:')
print('   ✅ Feature Engineering x2')
print('   ✅ Regularization')
print('   ✅ Feature Jittering (A/B тест)')
print('   ✅ Label Smoothing (A/B тест)')
print('   ✅ Ансамблирование (Ridge blending)')
print('   ✅ Инкремент baseline')

print(f'\n📊 РЕЗУЛЬТАТЫ:')
print(f'   Final R²: {ridge_r2:.6f}')
print(f'   Цель:     0.869')
print(f'   Статус:   {'✅ ДОСТИГНУТО!' if ridge_r2 >= 0.869 else '⚠️ Близко к цели'}')

print('\n📦 ФАЙЛЫ:')
print('   ✅ submission_advanced_winner.csv')
print('   ✅ advanced_hierarchical_clustering.png')
print('   ✅ advanced_ridge_weights.png')
print('   ✅ advanced_permutation_importance.png')

print('\n' + '='*70)
print('🚀 ADVANCED SOLUTION (WINNER METHODOLOGY) COMPLETED!')
print('='*70)